## 1. Importación de librerías

Se utilizan **Pandas** y **NumPy** para la manipulación y el análisis de datos, **Plotly** para la visualización interactiva 3D, y utilidades estándar (`json`, `pathlib`) para la construcción del panel interactivo con sprites.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, IFrame, HTML
import json
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
import plotly
print("Plotly:", plotly.__version__)


Pandas: 2.3.3
NumPy: 2.3.5
Plotly: 6.3.0


## 2. Carga del dataset

**Origen de los datos:** el archivo `Pokemon2.csv` contiene las estadísticas base de los
Pokémon de las **generaciones 1 a 6**, compiladas originalmente a partir de
[pokemondb.net](https://pokemondb.net) (metodología equivalente al repositorio público
[`lgreski/pokemonData`](https://github.com/lgreski/pokemonData)). El archivo incluye,
además, dos registros personalizados (`Sesni`, `Chuchin`) agregados como práctica del curso.

**Contenido:** 807 registros con el número de Pokédex, nombre, tipo primario y
secundario, seis estadísticas base (HP, Ataque, Defensa, Ataque especial, Defensa
especial y Velocidad), el total de estadísticas, la generación, si es legendario y su
nivel de evolución.

**Sprites:** las imágenes de cada Pokémon se obtienen del repositorio público
[`PokeAPI/sprites`](https://github.com/PokeAPI/sprites) en GitHub (carpeta
`sprites/pokemon/`), indexadas por el número de Pokédex nacional.


In [4]:
RUTA_ORIGEN = "Pokemon2.csv"
df_raw = pd.read_csv(RUTA_ORIGEN)
df_raw.head()


,dex_number,name,type_1,type_2,total,hp,attack,defense,sp_atk,sp_def,speed,generation,legendary,evolution_level,generacion_valida,promedio_estadisticas,sprite_url
0,16,Pidgey,Normal,Flying,251,40,45,40,35,35,56,1,False,1,True,41.83,https://raw.githubusercontent.com/PokeAPI/spri...
1,17,Pidgeotto,Normal,Flying,349,63,60,55,50,50,71,1,False,1,True,58.17,https://raw.githubusercontent.com/PokeAPI/spri...
2,18,Pidgeot,Normal,Flying,479,83,80,75,70,70,101,1,False,2,True,79.83,https://raw.githubusercontent.com/PokeAPI/spri...
3,18,PidgeotMega Pidgeot,Normal,Flying,579,83,80,80,135,80,121,1,False,3,True,96.50,https://raw.githubusercontent.com/PokeAPI/spri...
4,19,Rattata,Normal,Sin segundo tipo,253,30,56,35,25,35,72,1,False,1,True,42.17,https://raw.githubusercontent.com/PokeAPI/spri...


## 3. Inspección inicial del dataset

In [5]:
df_raw.head()


,dex_number,name,type_1,type_2,total,hp,attack,defense,sp_atk,sp_def,speed,generation,legendary,evolution_level,generacion_valida,promedio_estadisticas,sprite_url
0,16,Pidgey,Normal,Flying,251,40,45,40,35,35,56,1,False,1,True,41.83,https://raw.githubusercontent.com/PokeAPI/spri...
1,17,Pidgeotto,Normal,Flying,349,63,60,55,50,50,71,1,False,1,True,58.17,https://raw.githubusercontent.com/PokeAPI/spri...
2,18,Pidgeot,Normal,Flying,479,83,80,75,70,70,101,1,False,2,True,79.83,https://raw.githubusercontent.com/PokeAPI/spri...
3,18,PidgeotMega Pidgeot,Normal,Flying,579,83,80,80,135,80,121,1,False,3,True,96.50,https://raw.githubusercontent.com/PokeAPI/spri...
4,19,Rattata,Normal,Sin segundo tipo,253,30,56,35,25,35,72,1,False,1,True,42.17,https://raw.githubusercontent.com/PokeAPI/spri...


In [6]:
print("Dimensiones (filas, columnas):", df_raw.shape)


Dimensiones (filas, columnas): (803, 17)


In [7]:
df_raw.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 803 entries, 0 to 802
Data columns (total 17 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   dex_number             803 non-null    int64  
 1   name                   803 non-null    object 
 2   type_1                 803 non-null    object 
 3   type_2                 803 non-null    object 
 4   total                  803 non-null    int64  
 5   hp                     803 non-null    int64  
 6   attack                 803 non-null    int64  
 7   defense                803 non-null    int64  
 8   sp_atk                 803 non-null    int64  
 9   sp_def                 803 non-null    int64  
 10  speed                  803 non-null    int64  
 11  generation             803 non-null    int64  
 12  legendary              803 non-null    bool   
 13  evolution_level        803 non-null    int64  
 14  generacion_valida      803 non-null    bool   
 15  promed

In [8]:
df_raw.describe(include="all").T


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
dex_number,803.0,NaN,NaN,NaN,363.270237,209.099012,1.0,184.5,365.0,540.5,723.0
name,803,802,Caterpie,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
type_1,803,19,Water,112,NaN,NaN,NaN,NaN,NaN,NaN,NaN
type_2,803,19,Sin segundo tipo,387,NaN,NaN,NaN,NaN,NaN,NaN,NaN
total,803.0,NaN,NaN,NaN,435.058531,120.791407,180.0,330.0,450.0,515.0,787.0
hp,803.0,NaN,NaN,NaN,69.291407,25.569274,1.0,50.0,65.0,80.0,255.0
attack,803.0,NaN,NaN,NaN,79.84807,38.960743,5.0,55.0,75.0,100.0,676.0
defense,803.0,NaN,NaN,NaN,74.841843,38.663987,5.0,50.0,70.0,90.0,678.0
sp_atk,803.0,NaN,NaN,NaN,73.816936,40.147734,10.0,49.5,65.0,95.0,688.0
sp_def,803.0,NaN,NaN,NaN,72.890411,36.100724,20.0,50.0,70.0,90.0,678.0


## 4. Limpieza y normalización de columnas y categorías

Se normalizan los nombres de columnas (minúsculas, sin espacios ni puntos) y se
homogeneízan los valores de texto de las columnas categóricas (`Type 1`, `Type 2`),
eliminando espacios sobrantes y unificando el formato *Título*.


In [9]:
df = df_raw.copy()

# Normalización de nombres de columnas
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(".", "", regex=False)
    .str.replace(" ", "_")
)
df = df.rename(columns={"#": "dex_number"})
print("Columnas normalizadas:")
print(df.columns.tolist())


Columnas normalizadas:
['dex_number', 'name', 'type_1', 'type_2', 'total', 'hp', 'attack', 'defense', 'sp_atk', 'sp_def', 'speed', 'generation', 'legendary', 'evolution_level', 'generacion_valida', 'promedio_estadisticas', 'sprite_url']


In [10]:
# Normalización de valores categóricos (tipos de Pokémon)
for col in ["type_1", "type_2"]:
    df[col] = df[col].astype("string").str.strip().str.title()

df["name"] = df["name"].astype("string").str.strip()

print("Tipos únicos (type_1) tras normalizar:")
print(sorted(df["type_1"].unique()))


Tipos únicos (type_1) tras normalizar:
['Artificial', 'Bug', 'Dark', 'Dragon', 'Electric', 'Fairy', 'Fighting', 'Fire', 'Flying', 'Ghost', 'Grass', 'Ground', 'Ice', 'Normal', 'Poison', 'Psychic', 'Rock', 'Steel', 'Water']


## 5. Tratamiento de valores nulos, duplicados y datos incorrectos

**Antes de la limpieza:**


In [11]:
print("Filas antes de limpiar:", len(df))
print("\nValores nulos por columna (antes):")
print(df.isnull().sum())
print("\nRegistros duplicados exactos (antes):", df.duplicated().sum())


Filas antes de limpiar: 803

Valores nulos por columna (antes):
dex_number               0
name                     0
type_1                   0
type_2                   0
total                    0
hp                       0
attack                   0
defense                  0
sp_atk                   0
sp_def                   0
speed                    0
generation               0
legendary                0
evolution_level          0
generacion_valida        0
promedio_estadisticas    0
sprite_url               2
dtype: int64

Registros duplicados exactos (antes): 0


**Duplicados exactos:** se encontraron 4 filas totalmente duplicadas
(`Squirtle`, `Caterpie`, `Weedle`, `Butterfree`), producto de un error de carga del
archivo original. Se eliminan conservando la primera aparición.

**Valores nulos en `type_2`:** no son un error — representan Pokémon de un **solo
tipo**. Se sustituyen por la etiqueta explícita `"Sin segundo tipo"` en lugar de
eliminarlos, para no perder información de los Pokémon monotipo.

**Datos incorrectos (generación fuera de rango):** se detectaron **2 registros**
(`Sesni` #722 y `Chuchin` #723) con generación `67` y `32` respectivamente y un tipo
`"Artificial"` que no pertenece al roster canónico de tipos Pokémon. Dado que el rango
válido de generaciones en este dataset es 1–6, se marcan como registros **especiales**
(no se eliminan, para preservar la información), pero se **excluyen del eje de
generación** en la visualización principal.


In [12]:
# --- Duplicados ---
filas_antes = len(df)
duplicados = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)
print(f"Duplicados eliminados: {duplicados}  |  Filas: {filas_antes} -> {len(df)}")

# --- Nulos en type_2 ---
nulos_antes = df["type_2"].isnull().sum()
df["type_2"] = df["type_2"].fillna("Sin segundo tipo")
print(f"Nulos en type_2: {nulos_antes} -> {df['type_2'].isnull().sum()}")

# --- Datos incorrectos: generaciones fuera de rango ---
generaciones_validas = list(range(1, 7))
df["generacion_valida"] = df["generation"].isin(generaciones_validas)
print("\nRegistros con generación fuera de rango (1-6):")
display(df.loc[~df["generacion_valida"], ["dex_number","name","type_1","generation","legendary"]])


Duplicados eliminados: 0  |  Filas: 803 -> 803
Nulos en type_2: 0 -> 0

Registros con generación fuera de rango (1-6):


,dex_number,name,type_1,generation,legendary
801,723,Chuchin,Artificial,32,True
802,722,Sesni,Artificial,67,True


In [13]:
print("Estado final:")
print("Filas:", len(df))
print("Nulos totales:", df.isnull().sum().sum())
print("Duplicados restantes:", df.duplicated().sum())


Estado final:
Filas: 803
Nulos totales: 2
Duplicados restantes: 0


## 6. Selección de variables estadísticas

Se seleccionan las **seis estadísticas base** oficiales de cada Pokémon —
`hp`, `attack`, `defense`, `sp_atk`, `sp_def` y `speed` — porque son las variables
numéricas fundamentales que determinan el desempeño de un Pokémon en batalla y son,
además, los componentes que originalmente conforman la columna `total`. Usarlas de
forma individual (en lugar de solo `total`) permite construir un promedio interpretable
y compararlas entre sí más adelante.


In [14]:
stats_cols = ["hp", "attack", "defense", "sp_atk", "sp_def", "speed"]
df[["name"] + stats_cols + ["total"]].head()


,name,hp,attack,defense,sp_atk,sp_def,speed,total
0,Pidgey,40,45,40,35,35,56,251
1,Pidgeotto,63,60,55,50,50,71,349
2,Pidgeot,83,80,75,70,70,101,479
3,PidgeotMega Pidgeot,83,80,80,135,80,121,579
4,Rattata,30,56,35,25,35,72,253


## 7. Creación de la columna `promedio_estadisticas`

In [15]:
df["promedio_estadisticas"] = df[stats_cols].mean(axis=1).round(2)
df[["name", "type_1", "generation"] + stats_cols + ["promedio_estadisticas"]].head()


,name,type_1,generation,hp,attack,defense,sp_atk,sp_def,speed,promedio_estadisticas
0,Pidgey,Normal,1,40,45,40,35,35,56,41.83
1,Pidgeotto,Normal,1,63,60,55,50,50,71,58.17
2,Pidgeot,Normal,1,83,80,75,70,70,101,79.83
3,PidgeotMega Pidgeot,Normal,1,83,80,80,135,80,121,96.50
4,Rattata,Normal,1,30,56,35,25,35,72,42.17


## 8. Análisis estadístico descriptivo

In [17]:
dfv = df[df["generacion_valida"]].copy()  # se excluyen los 2 registros especiales

resumen = dfv[stats_cols + ["promedio_estadisticas"]].agg(["mean", "median", "min", "max", "std"]).T
resumen.columns = ["media", "mediana", "minimo", "maximo", "desv_estandar"]
resumen.round(2)


,media,mediana,minimo,maximo,desv_estandar
hp,69.23,65.0,1.0,255.0,25.53
attack,78.94,75.0,5.0,190.0,32.48
defense,73.79,70.0,5.0,230.0,31.19
sp_atk,72.75,65.0,10.0,194.0,32.76
sp_def,71.84,70.0,20.0,230.0,27.87
speed,68.25,65.0,5.0,180.0,29.05
promedio_estadisticas,72.47,75.0,30.0,130.0,20.03


In [18]:
print("Promedio de estadísticas por generación:")
display(dfv.groupby("generation")["promedio_estadisticas"].mean().round(2))

print("\nPromedio de estadísticas por tipo principal (top 5):")
display(dfv.groupby("type_1", observed=True)["promedio_estadisticas"]
        .mean().round(2).sort_values(ascending=False).head())

print("\nLegendarios vs. no legendarios:")
display(dfv.groupby("legendary")["promedio_estadisticas"].mean().round(2))


Promedio de estadísticas por generación:


generation
1    70.90
2    69.71
3    72.70
4    76.50
5    72.50
6    72.73
Name: promedio_estadisticas, dtype: float64


Promedio de estadísticas por tipo principal (top 5):


type_1
Dragon     91.76
Steel      81.28
Flying     80.84
Psychic    79.32
Fire       76.35
Name: promedio_estadisticas, dtype: float64


Legendarios vs. no legendarios:


legendary
False     69.54
True     105.11
Name: promedio_estadisticas, dtype: float64

## 9. Preparación y orden de las variables `generación` y `tipo`

Se define un orden categórico para `type_1` (siguiendo el orden tradicional de la
Pokédex) para que la visualización agrupe los tipos de forma consistente en el eje Y,
y se ordena el DataFrame por generación, tipo y número de Pokédex.


In [19]:
orden_tipos = [
    "Normal", "Fire", "Water", "Electric", "Grass", "Ice", "Fighting",
    "Poison", "Ground", "Flying", "Psychic", "Bug", "Rock", "Ghost",
    "Dragon", "Dark", "Steel", "Fairy",
]
tipos_presentes = [t for t in orden_tipos if t in dfv["type_1"].unique()]

dfv["type_1"] = pd.Categorical(dfv["type_1"], categories=tipos_presentes, ordered=True)
dfv = dfv.sort_values(["generation", "type_1", "dex_number"]).reset_index(drop=True)

print("Orden de tipos utilizado en el eje Y:")
print(tipos_presentes)


Orden de tipos utilizado en el eje Y:
['Normal', 'Fire', 'Water', 'Electric', 'Grass', 'Ice', 'Fighting', 'Poison', 'Ground', 'Flying', 'Psychic', 'Bug', 'Rock', 'Ghost', 'Dragon', 'Dark', 'Steel', 'Fairy']


## 10. Obtención y validación de los sprites

Los sprites se construyen a partir del número de Pokédex (`dex_number`), apuntando al
repositorio [`PokeAPI/sprites`](https://github.com/PokeAPI/sprites):

```
https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/{dex_number}.png
```

Los dos registros especiales (`Sesni`, `Chuchin`) no cuentan con un sprite oficial real
(usar su número de Pokédex mostraría, incorrectamente, el sprite de un Pokémon de la
generación 7), por lo que se les asigna `None` y la visualización los reemplaza por un
ícono de marcador de posición (❔).


In [20]:
SPRITE_BASE = "https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/{}.png"

def construir_sprite(row):
    if not row["generacion_valida"]:
        return None
    return SPRITE_BASE.format(int(row["dex_number"]))

dfv["sprite_url"] = dfv.apply(construir_sprite, axis=1)
dfv[["dex_number", "name", "sprite_url"]].head()


,dex_number,name,sprite_url
0,16,Pidgey,https://raw.githubusercontent.com/PokeAPI/spri...
1,17,Pidgeotto,https://raw.githubusercontent.com/PokeAPI/spri...
2,18,Pidgeot,https://raw.githubusercontent.com/PokeAPI/spri...
3,18,PidgeotMega Pidgeot,https://raw.githubusercontent.com/PokeAPI/spri...
4,19,Rattata,https://raw.githubusercontent.com/PokeAPI/spri...


In [21]:
# Validación de accesibilidad de una muestra de sprites
import requests
muestra = dfv["sprite_url"].dropna().sample(5, random_state=42)
for url in muestra:
    try:
        r = requests.head(url, timeout=10)
        print(r.status_code, url)
    except Exception as e:
        print("ERROR", url, e)


200 https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/643.png
200 https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/544.png
200 https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/121.png
200 https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/438.png
200 https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/130.png


## 11. Primera versión del Scatter Plot 3D

Se construye una primera versión con `plotly.express.scatter_3d`, usando la
**generación** en el eje X, el **tipo** en el eje Y y el **promedio de estadísticas**
en el eje Z, coloreando por tipo.


In [22]:
dfv["type_1_str"] = dfv["type_1"].astype(str)

fig_v1 = px.scatter_3d(
    dfv,
    x="generation",
    y="type_1_str",
    z="promedio_estadisticas",
    color="type_1_str",
    hover_name="name",
    category_orders={"type_1_str": tipos_presentes},
    title="Versión 1 · Scatter 3D: Generación, Tipo y Promedio de estadísticas",
    labels={"generation": "Generación", "type_1_str": "Tipo", "promedio_estadisticas": "Promedio de estadísticas"},
    opacity=0.85,
    height=650,
)
fig_v1.update_traces(marker=dict(size=4))
fig_v1.show()


## 12. Diferenciación visual por tipo (color y símbolo)

Se asigna a cada tipo un **color** (paleta oficial aproximada de tipos Pokémon) y un
**símbolo** de marcador distinto (ciclando entre los símbolos disponibles en
`scatter3d`, ya que Plotly solo ofrece 8 símbolos 3D nativos frente a 18 tipos), de
forma que la combinación color+símbolo permita distinguir cada tipo incluso en
escala de grises.


In [23]:
COLOR_TIPO = {
    "Normal": "#A8A878", "Fire": "#F08030", "Water": "#6890F0",
    "Electric": "#F8D030", "Grass": "#78C850", "Ice": "#98D8D8",
    "Fighting": "#C03028", "Poison": "#A040A0", "Ground": "#E0C068",
    "Flying": "#A890F0", "Psychic": "#F85888", "Bug": "#A8B820",
    "Rock": "#B8A038", "Ghost": "#705898", "Dragon": "#7038F8",
    "Dark": "#705848", "Steel": "#B8B8D0", "Fairy": "#EE99AC",
}
SIMBOLOS = ["circle", "diamond", "square", "cross", "x", "circle-open", "diamond-open", "square-open"]
SIMBOLO_TIPO = {t: SIMBOLOS[i % len(SIMBOLOS)] for i, t in enumerate(tipos_presentes)}

dfv["color"] = dfv["type_1_str"].map(COLOR_TIPO)
dfv["simbolo"] = dfv["type_1_str"].map(SIMBOLO_TIPO)

pd.DataFrame({"tipo": tipos_presentes,
              "color": [COLOR_TIPO[t] for t in tipos_presentes],
              "simbolo": [SIMBOLO_TIPO[t] for t in tipos_presentes]})



,tipo,color,simbolo
0,Normal,#A8A878,circle
1,Fire,#F08030,diamond
2,Water,#6890F0,square
3,Electric,#F8D030,cross
4,Grass,#78C850,x
5,Ice,#98D8D8,circle-open
6,Fighting,#C03028,diamond-open
7,Poison,#A040A0,square-open
8,Ground,#E0C068,circle
9,Flying,#A890F0,diamond


## 13. Información emergente (hover) y sprites integrados

La versión final de la visualización se construye como una página HTML autónoma
(Plotly.js + JavaScript) que:

- Muestra en el **hover** de cada punto: nombre, tipo, generación y promedio de
  estadísticas.
- Incorpora un **panel lateral interactivo** que, al pasar el cursor sobre un punto,
  despliega el **sprite** del Pokémon (obtenido de `PokeAPI/sprites`), su tipo (con
  insignia de color), si es legendario y una barra por cada estadística base.
- Incluye **filtros interactivos** por generación (botones), por tipo (chips —también
  funcionan como leyenda) y por rango de promedio de estadísticas (doble control
  deslizante).
- Personaliza título, etiquetas de los ejes, tamaño, opacidad, cámara 3D y
  proporción de aspecto (`aspectratio`).

A continuación se genera y se muestra esa visualización interactiva:


In [28]:
# Construcción del archivo HTML interactivo final
# (usa dfv, tipos_presentes, COLOR_TIPO y SIMBOLO_TIPO ya calculados arriba)

dfv["sprite_url"] = dfv["sprite_url"].fillna("")

puntos = []
for _, r in dfv.iterrows():
    puntos.append({
        "x": int(r["generation"]),
        "y": tipos_presentes.index(r["type_1_str"]),
        "z": float(r["promedio_estadisticas"]),
        "name": r["name"],
        "type1": r["type_1_str"],
        "type2": r["type_2"],
        "gen": int(r["generation"]),
        "prom": float(r["promedio_estadisticas"]),
        "hp": int(r["hp"]), "atk": int(r["attack"]), "df": int(r["defense"]),
        "spa": int(r["sp_atk"]), "spd": int(r["sp_def"]), "spe": int(r["speed"]),
        "legendary": bool(r["legendary"]),
        "sprite": r["sprite_url"],
        "color": r["color"],
        "simbolo": r["simbolo"],
    })

data_json = json.dumps(puntos)
tipos_json = json.dumps(tipos_presentes)
colores_json = json.dumps(COLOR_TIPO)

print("Puntos preparados para la visualización:", len(puntos))

html_template = """<!DOCTYPE html>
<html lang="es">
<head>
<meta charset="utf-8" />
<title>Scatter 3D de Pok\u00e9mon: Promedio de Estad\u00edsticas por Tipo y Generaci\u00f3n</title>
<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
<style>
  :root {{
    --bg: #f4f6fb; --panel: #ffffff; --ink: #1f2430; --muted: #6b7280;
    --accent: #5b6ee1; --border: #e3e6f0;
  }}
  * {{ box-sizing: border-box; }}
  body {{
    margin: 0; font-family: 'Segoe UI', system-ui, sans-serif;
    background: var(--bg); color: var(--ink);
  }}
  header {{ padding: 18px 24px 6px 24px; }}
  header h1 {{ font-size: 20px; margin: 0 0 4px 0; }}
  header p {{ margin: 0; color: var(--muted); font-size: 13px; }}
  #layout {{
    display: grid; grid-template-columns: 1fr 300px; gap: 16px;
    padding: 12px 24px 24px 24px; align-items: start;
  }}
  #controls {{
    background: var(--panel); border: 1px solid var(--border); border-radius: 12px;
    padding: 14px 16px; margin: 0 24px 8px 24px;
    display: flex; flex-wrap: wrap; gap: 18px; align-items: center;
  }}
  .ctrl-group {{ display:flex; flex-direction:column; gap:4px; font-size:12px; color: var(--muted); }}
  .ctrl-group label.title {{ font-weight: 600; color: var(--ink); }}
  select, input[type=range] {{ font-family: inherit; }}
  #gen-buttons button {{
    border: 1px solid var(--border); background: #fff; border-radius: 8px;
    padding: 4px 10px; margin-right: 4px; cursor: pointer; font-size: 12px;
  }}
  #gen-buttons button.active {{ background: var(--accent); color: #fff; border-color: var(--accent); }}
  #chip-legend {{ display:flex; flex-wrap: wrap; gap: 6px; max-width: 480px; }}
  .chip {{
    display:flex; align-items:center; gap:5px; border:1px solid var(--border);
    border-radius: 999px; padding: 3px 9px 3px 7px; font-size: 11px; cursor: pointer;
    background: #fff; user-select:none;
  }}
  .chip .dot {{ width:9px; height:9px; border-radius:50%; }}
  .chip.inactive {{ opacity: 0.35; }}
  #range-vals {{ font-size: 11px; color: var(--muted); }}
  #plot {{ height: 640px; background: var(--panel); border-radius: 12px; border: 1px solid var(--border); }}
  #sidepanel {{
    background: var(--panel); border: 1px solid var(--border); border-radius: 12px;
    padding: 16px; min-height: 640px;
  }}
  #sidepanel h3 {{ margin: 0 0 10px 0; font-size: 14px; color: var(--muted); font-weight: 600; }}
  #sprite-box {{
    width: 100%; height: 140px; display: flex; align-items: center; justify-content: center;
    background: #f0f2fa; border-radius: 10px; margin-bottom: 12px;
  }}
  #sprite-box img {{ max-height: 120px; image-rendering: pixelated; }}
  #sprite-placeholder {{ font-size: 42px; color: #c6cbe0; }}
  #info-name {{ font-size: 18px; font-weight: 700; margin: 0 0 6px 0; }}
  .info-row {{ display:flex; justify-content: space-between; font-size: 13px; padding: 3px 0; border-bottom: 1px dashed var(--border); }}
  .info-row span:first-child {{ color: var(--muted); }}
  .type-badge {{ display:inline-block; padding: 2px 8px; border-radius: 6px; color:#fff; font-size:11px; font-weight:600; margin-right:4px;}}
  #stats-mini {{ margin-top: 10px; }}
  .bar-row {{ display:flex; align-items:center; gap:6px; font-size: 11px; margin: 3px 0; }}
  .bar-row .lbl {{ width: 38px; color: var(--muted); }}
  .bar-track {{ flex:1; background:#eef0f8; border-radius:4px; height:7px; overflow:hidden; }}
  .bar-fill {{ height:100%; background: var(--accent); }}
  #empty-msg {{ color: var(--muted); font-size: 13px; text-align:center; margin-top: 40px; }}
  footer {{ padding: 0 24px 24px 24px; color: var(--muted); font-size: 11px; }}
</style>
</head>
<body>
<header>
  <h1>Scatter 3D de Pok\u00e9mon: Promedio de Estad\u00edsticas por Tipo y Generaci\u00f3n</h1>
  <p>Pr\u00e1ctica 07 \u00b7 Cada punto es un Pok\u00e9mon \u00b7 La posici\u00f3n Z representa el promedio de sus estad\u00edsticas base (HP, Ataque, Defensa, At. Especial, Def. Especial, Velocidad)</p>
</header>

<div id="controls">
  <div class="ctrl-group">
    <label class="title">Generaci\u00f3n</label>
    <div id="gen-buttons"></div>
  </div>
  <div class="ctrl-group" style="max-width:520px;">
    <label class="title">Tipo (clic para filtrar / color y s\u00edmbolo)</label>
    <div id="chip-legend"></div>
  </div>
  <div class="ctrl-group">
    <label class="title">Rango de promedio de estad\u00edsticas</label>
    <input type="range" id="min-range" min="0" max="140" value="0" step="1">
    <input type="range" id="max-range" min="0" max="140" value="140" step="1">
    <div id="range-vals">0 \u2013 140</div>
  </div>
</div>

<div id="layout">
  <div id="plot"></div>
  <div id="sidepanel">
    <h3>Detalle del Pok\u00e9mon</h3>
    <div id="detail-content">
      <div id="empty-msg">Pasa el cursor sobre un punto de la gr\u00e1fica<br>para ver su sprite y sus estad\u00edsticas.</div>
    </div>
  </div>
</div>
<footer>Fuente de datos: pokemondb.net (v\u00eda lgreski/pokemonData) \u00b7 Sprites: PokeAPI/sprites (GitHub)</footer>

<script>
const DATA = {data_json};
const TIPOS = {tipos_json};
const COLOR_TIPO = {colores_json};

let filtroTipos = new Set(TIPOS);
let filtroGen = "all";
let minZ = 0, maxZ = 140;

function datosFiltrados() {{
  return DATA.filter(p =>
    filtroTipos.has(p.type1) &&
    (filtroGen === "all" || p.gen === filtroGen) &&
    p.prom >= minZ && p.prom <= maxZ
  );
}}

function construirTrazo(puntos) {{
  return [{{
    type: "scatter3d",
    mode: "markers",
    x: puntos.map(p => p.x),
    y: puntos.map(p => p.y),
    z: puntos.map(p => p.z),
    marker: {{
      size: 6,
      opacity: 0.9,
      color: puntos.map(p => p.color),
      symbol: puntos.map(p => p.simbolo),
      line: {{ width: puntos.map(p => p.legendary ? 2 : 0), color: "#222" }}
    }},
    customdata: puntos.map(p => p),
    hovertemplate: "<b>%{{customdata.name}}</b><br>" +
                    "Tipo: %{{customdata.type1}}<br>" +
                    "Generaci\u00f3n: %{{customdata.gen}}<br>" +
                    "Promedio: %{{customdata.prom}}<extra></extra>",
  }}];
}}

const layout = {{
  title: {{ text: "" }},
  paper_bgcolor: "rgba(0,0,0,0)",
  plot_bgcolor: "rgba(0,0,0,0)",
  margin: {{ l: 0, r: 0, t: 10, b: 0 }},
  scene: {{
    xaxis: {{ title: "Generaci\u00f3n", tickvals: [1,2,3,4,5,6], gridcolor: "#e3e6f0" }},
    yaxis: {{ title: "Tipo", tickvals: TIPOS.map((t,i)=>i), ticktext: TIPOS, gridcolor: "#e3e6f0" }},
    zaxis: {{ title: "Promedio de estad\u00edsticas", gridcolor: "#e3e6f0" }},
    aspectmode: "manual",
    aspectratio: {{ x: 1.1, y: 1.6, z: 1 }},
    camera: {{ eye: {{ x: 1.55, y: 1.55, z: 0.9 }} }},
    bgcolor: "rgba(0,0,0,0)",
  }},
  showlegend: false,
}};

const config = {{ responsive: true, displaylogo: false }};

Plotly.newPlot("plot", construirTrazo(datosFiltrados()), layout, config);

function actualizarGrafico() {{
  const puntos = datosFiltrados();
  Plotly.react("plot", construirTrazo(puntos), layout, config);
}}

const genBox = document.getElementById("gen-buttons");
["all",1,2,3,4,5,6].forEach(g => {{
  const b = document.createElement("button");
  b.textContent = g === "all" ? "Todas" : "Gen " + g;
  if (g === filtroGen) b.classList.add("active");
  b.onclick = () => {{
    filtroGen = g;
    [...genBox.children].forEach(c => c.classList.remove("active"));
    b.classList.add("active");
    actualizarGrafico();
  }};
  genBox.appendChild(b);
}});

const chipBox = document.getElementById("chip-legend");
TIPOS.forEach(t => {{
  const c = document.createElement("div");
  c.className = "chip";
  c.innerHTML = `<span class="dot" style="background:${{COLOR_TIPO[t]}}"></span>${{t}}`;
  c.onclick = () => {{
    if (filtroTipos.has(t)) {{ filtroTipos.delete(t); c.classList.add("inactive"); }}
    else {{ filtroTipos.add(t); c.classList.remove("inactive"); }}
    actualizarGrafico();
  }};
  chipBox.appendChild(c);
}});

const minR = document.getElementById("min-range");
const maxR = document.getElementById("max-range");
const rangeVals = document.getElementById("range-vals");
function actualizarRango() {{
  let a = parseInt(minR.value), b = parseInt(maxR.value);
  if (a > b) {{ [a,b] = [b,a]; }}
  minZ = a; maxZ = b;
  rangeVals.textContent = a + " \u2013 " + b;
  actualizarGrafico();
}}
minR.addEventListener("input", actualizarRango);
maxR.addEventListener("input", actualizarRango);

const detail = document.getElementById("detail-content");
function statBar(lbl, val) {{
  const pct = Math.min(100, (val/200)*100);
  return `<div class="bar-row"><div class="lbl">${{lbl}}</div><div class="bar-track"><div class="bar-fill" style="width:${{pct}}%"></div></div><div>${{val}}</div></div>`;
}}
function mostrarDetalle(p) {{
  const spriteHtml = p.sprite
    ? `<img src="${{p.sprite}}" alt="${{p.name}}">`
    : `<span id="sprite-placeholder">\u2754</span>`;
  detail.innerHTML = `
    <div id="sprite-box">${{spriteHtml}}</div>
    <p id="info-name">${{p.name}}</p>
    <span class="type-badge" style="background:${{COLOR_TIPO[p.type1]||'#999'}}">${{p.type1}}</span>
    ${{p.type2 && p.type2 !== 'Sin segundo tipo' ? `<span class="type-badge" style="background:${{COLOR_TIPO[p.type2]||'#888'}}">${{p.type2}}</span>` : ""}}
    <div class="info-row"><span>Generaci\u00f3n</span><span>${{p.gen}}</span></div>
    <div class="info-row"><span>Promedio de estad\u00edsticas</span><span>${{p.prom}}</span></div>
    <div class="info-row"><span>Legendario</span><span>${{p.legendary ? "S\u00ed" : "No"}}</span></div>
    <div id="stats-mini">
      ${{statBar("HP", p.hp)}}
      ${{statBar("Atq", p.atk)}}
      ${{statBar("Def", p.df)}}
      ${{statBar("AtE", p.spa)}}
      ${{statBar("DfE", p.spd)}}
      ${{statBar("Vel", p.spe)}}
    </div>`;
}}

const plotDiv = document.getElementById("plot");
plotDiv.on("plotly_hover", (ev) => {{
  const p = ev.points[0].customdata;
  mostrarDetalle(p);
}});
</script>
</body>
</html>
""".format(data_json=data_json, tipos_json=tipos_json, colores_json=colores_json)

with open("practica07_scatter3d_pokemon.html", "w", encoding="utf-8") as f:
    f.write(html_template)

print("Archivo HTML generado:", len(html_template), "caracteres")


Puntos preparados para la visualización: 801
Archivo HTML generado: 280679 caracteres


In [29]:
IFrame(src="practica07_scatter3d_pokemon.html", width="100%", height=780)


## 14. Exportación de la visualización en HTML

La visualización ya fue exportada como archivo HTML autónomo
(`practica07_scatter3d_pokemon.html`), incluyendo Plotly.js vía CDN. El archivo
conserva **todas** sus funciones interactivas (rotación 3D, zoom, hover con sprite,
filtros por generación/tipo/rango) al abrirse directamente en cualquier navegador,
sin depender de Jupyter.


In [30]:
ruta_html = Path("practica07_scatter3d_pokemon.html")
print("Archivo exportado:", ruta_html.resolve())
print("Tamaño:", round(ruta_html.stat().st_size / 1024, 1), "KB")
print("Existe:", ruta_html.exists())


Archivo exportado: C:\Users\derek\Desktop\ECBD_9A_IDGS_PRACTICAS_230892\Practica08\practica07_scatter3d_pokemon.html
Tamaño: 274.4 KB
Existe: True


## 15. Hallazgos e interpretación

**1. Los Dragón y legendarios dominan las estadísticas más altas.**
El tipo `Dragon` tiene, en promedio, las estadísticas más altas de todos los tipos
principales (~91.8 de promedio, frente a ~72.5 del promedio general), y los Pokémon
`legendary=True` promedian **105.1** frente a **69.5** de los no legendarios — una
brecha de más de 35 puntos que confirma el diseño intencional de los legendarios como
Pokémon superiores en estadísticas.

**2. Las formas Mega y Primal generan los valores atípicos superiores.**
Los cinco Pokémon con mayor promedio de estadísticas (`Mega Mewtwo X/Y`, `Mega
Rayquaza`, `Primal Kyogre`, `Primal Groudon`) son todas variantes *Mega/Primal*, no
Pokémon base. Esto se observa claramente como un grupo de puntos que se separa del
resto en la parte superior del gráfico 3D, indicando que estas mecánicas de juego
(introducidas en generaciones 3 y 6) están diseñadas para maximizar el poder de
combate temporalmente.

**3. La generación 4 tiene, en promedio, las estadísticas base más altas.**
Al agrupar por generación, la Generación 4 (`Diamond`/`Pearl`/`Platinum`) presenta el
promedio más alto (76.5), seguida de la 6 y la 3 (~72.7), mientras que la Generación 2
tiene el promedio más bajo (69.7) — un patrón conocido como *power creep*, donde
generaciones más recientes tienden a introducir Pokémon con estadísticas base
ligeramente mayores.

**4. Dos registros son atípicos por diseño, no por error de captura.**
Los registros `Sesni` (#722) y `Chuchin` (#723) presentan estadísticas fuera de todo
rango Pokémon real (ataque de 676 y defensa de 678, cuando el máximo real observado en
el dataset es 190 y 230 respectivamente) y generaciones inexistentes (67 y 32). Se
identificaron como registros personalizados/de práctica y se trataron por separado
para no distorsionar el análisis de generaciones y tipos oficiales.
